In [ ]:
import numpy as np
from aed_rate.electronic.potential import create_oh_system_acharya, create_lih_system
from aed_rate.electronic.wavefunctions import ElectronicStructure
from aed_rate.electronic.coupling import ElectronicCoupling, InterpolatedCoupling
from aed_rate.nuclear.nuclear_wavefunction import create_wavefunction_solver
from aed_rate.utils.constants import CONSTANTS, get_reduced_mass
from aed_rate.utils import plotting

In [ ]:
anion, neutral, EA = create_lih_system()
mu = get_reduced_mass("O", "H")
E  = CONSTANTS.cm1_to_hartree(66.0)

In [ ]:
# 1. Potential energy curves
plotting.plot_potential_curves(anion, neutral, EA, reduced_mass=mu, J_values=[20, 50], R_range=[1, 15])

In [ ]:

# 2. Nuclear wavefunctions (bound χ_{v'}, scattering F_E, and dF_E/dR)
solver = create_wavefunction_solver(anion, mu, method="morse", n_grid=6000)
bound  = solver.solve_bound_state(v=0, J=0)
scatt  = solver.solve_scattering_state(E, J=0, normalization="unit_amplitude")
dF_dR  = solver.wavefunction_derivative(scatt)
plotting.plot_bound_states(solver.solve_all_bound_states(J=0), anion)
plotting.plot_scattering_derivative(scatt, dF_dR)

In [ ]:
# 3. Electronic coupling m_rad(R), m_rot(R) and its real-space ingredients
ele_str = ElectronicStructure("Li", "H", basis="6-311+G**")
coupling = ElectronicCoupling(ele_str, homo_symmetry="sigma")
m   = coupling.compute_coupling_at_r(R=3.2, electron_energy=0.001, spin=1, charge=-1)   # m_rad, m_rot
ing = coupling.compute_coupling_intermediates(R=3.2, electron_energy=0.001, spin=1, charge=-1)
plotting.plot_electronic_intermediates(ing)        # ∂φ_HOMO/∂R and the OPW φ_k

In [ ]:
# prcompute couplings
coupling = InterpolatedCoupling(ele_str, anion_potential=anion,  homo_symmetry="sigma",  spin=1)
coupling.precompute()
coupling.save("lih_coupling.npz")

In [ ]:
# The coupling integrand χ·m_rad·dF_E/dR (and its cancellation)

# prcompute couplings
#coupling = InterpolatedCoupling(ele_str, anion_potential=anion,  homo_symmetry="sigma",  spin=1)
#coupling.precompute()
#coupling.save("lih_coupling.npz")

coupling = InterpolatedCoupling.from_npz("lih_coupling.npz")
m_rad_R = np.array([coupling.compute_coupling_at_r(R, 0.01).m_rad.real
                    for R in solver.r_grid])

plotting.plot_coupling_integrand(solver.r_grid, bound.wavefunction, m_rad_R, dF_dR)

In [ ]:
# ============================================================
#  Cross sections & state-to-state exploration
# ============================================================
from pathlib import Path
from aed_rate.aed_calculator import AEDSystem

# Coupling that INCLUDES the l=0 s-wave
_npz = next(p for p in ("../lih_minus_coupling_swave.npz",
                        "lih_minus_coupling_swave.npz") if Path(p).exists())
coupling = InterpolatedCoupling.from_npz(_npz)
assert coupling.swave_channel is not None, "this npz has no s-wave -- re-precompute"
print("loaded", _npz, "| swave_channel =", coupling.swave_channel)

mu = get_reduced_mass("Li", "H")          # NB: cell [1] used ("O","H") by mistake
system = AEDSystem(
    anion_potential=anion, neutral_potential=neutral, EA=EA,
    reduced_mass=mu, coupling=coupling,
    solver_method="morse",                # required for cross sections
    n_grid=6000,                          # dense grid -> converged coupling integral
)
system

In [ ]:
# --- State-to-state cross section  sigma_{v',J->J'}(E) -------------------
# The facade returns just the number; the low-level calculator returns the
# full CrossSection object (electron energy + the three coupling integrals).
E = CONSTANTS.cm1_to_hartree(66.0)        # collision energy (Hartree)
# For a sigma-HOMO the s-wave is in the RADIAL channel (Delta J = 0),
# so J' = J shows V_swave != 0; J' = J +/- 1 (rotational) is l=1 only.
J, v_prime, J_prime = 0, 6, 0

cs = system._rate_calc.cross_section_state_to_state(E, J, v_prime, J_prime)
print(f"sigma            = {cs.sigma:.4e} a0^2  "
      f"({cs.sigma*CONSTANTS.bohr_to_angstrom**2:.4e} A^2)")
print(f"electron energy  = {cs.electron_energy*CONSTANTS.hartree_to_ev:.3f} eV")
print(f"V_rad = {cs.V_rad:+.3e}   (l=1 radial)")
print(f"V_rot = {cs.V_rot:+.3e}   (l=1 rotational)")
print(f"V_sw  = {cs.V_swave:+.3e}   (l=0 s-wave, channel='{coupling.swave_channel}')")

# Facade shortcut: omit J_prime to sum over J' in {J-1, J, J+1}
system.cross_section(E, J=0, v_prime=6)   # a0^2

In [ ]:
# --- Cross section resolved by final vibrational state v' ----------------
import matplotlib.pyplot as plt

vmax  = len(system.neutral_bound_states(J=0)) - 1
sig_v = {v: system.cross_section(E, J=0, v_prime=v) for v in range(vmax + 1)}

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(list(sig_v), list(sig_v.values()), color="C0")
ax.set_xlabel("final vibrational state  v'")
ax.set_ylabel(r"$\sigma$  ($a_0^2$)")
ax.set_title(f"LiH- AED cross section by v'  "
             f"(E = {E*CONSTANTS.hartree_to_cm1:.0f} cm^-1, J = 0)")
plt.show()
print("peak v' =", max(sig_v, key=sig_v.get))

In [ ]:
# --- Total sigma_AD(E): summed over J, v', J', and electron partial wave --
eV    = CONSTANTS.hartree_to_ev
E_meV = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500])
sigma_AD = np.array([system.sigma_AD(e * 1e-3 / eV, "a0^2") for e in E_meV])

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(E_meV, sigma_AD, "o-", color="C3")
ax.set_xlabel("collision energy  (meV)")
ax.set_ylabel(r"$\sigma_{AD}$  ($a_0^2$)")
ax.grid(True, which="both", alpha=0.3)
plt.show()